## Generate psychometric curves for different states

In [ ]:
import numpy as np
from glm_hmm_utils import *
from neurodatatypes import Session
import matplotlib.pyplot as plt
from pandas import read_pickle
import matplotlib
from pathlib import Path

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams.update({'font.size': 18})

### Get the data

In [ ]:
MODEL_PATH = r'X:\Widefield\glm_hmm_models\map_all_subjects_targrate.pickle'
#MODEL_PATH = r'X:\Widefield\glm_hmm_models\fez_4_state.pickle'
FIGSAVE_PATH = Path(r'C:\Data\churchland\state_manuscript_new_figs\raw_figures\glmhmm_new')

SAVE_FIGS = True

f = read_pickle(MODEL_PATH)
coherence = [-inpts[:,0] for inpts in f.inpts] # undo the sign flip from model fitting
all_coherence = np.hstack(coherence)

all_coherence = all_coherence * 20

all_choices = np.squeeze(np.vstack(f.outputs)).astype(bool)
print(all_choices)

mdl, ind, _ = f.return_best_model('ll',f.inpts, f.outputs)
states = np.vstack(f.return_ordered_states(modelindex=ind))

print(np.unique(all_coherence))

assert all_choices.size == all_coherence.size == states.shape[0]

print(f'There are {all_choices.size} trials')

### Code for psychometric func fitting

In [ ]:
p_right, ci_right = compute_p_right(all_coherence,all_choices)

In [ ]:
# run MLE
func = lambda pars: neg_log_likelihood_error(cumulative_gaussian, pars, all_coherence.astype(float),all_choices.astype(float))
# x0 is the initial guess for the fit, it is an important parameter
x0 = [0.,0.1,p_right[0],1 - p_right[-1]]

coherence_set = np.unique(all_coherence)
bounds = [(coherence_set[0],coherence_set[-1]),(0.0001,10),(0,0.7),(0,.7)]

res = minimize(func, x0, options = dict(maxiter = 500*len(x0),adaptive=True),
               bounds = bounds, method='Nelder-Mead') # method = 'L-BFGS-B',
print(res.x)

In [ ]:
# plotting util from Lukas Oesch
def separate_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    yti = ax.get_yticks()
    yti = yti[(yti >= ax.get_ylim()[0]) & (yti <= ax.get_ylim()[1]+10**-4)] #Add a small value to cover for some very tiny added values
    ax.spines['left'].set_bounds([yti[0], yti[-1]])
    xti = ax.get_xticks()
    xti = xti[(xti >= ax.get_xlim()[0]) & (xti <= ax.get_xlim()[1]+10**-4)]
    ax.spines['bottom'].set_bounds([xti[0], xti[-1]])
    return

In [ ]:
nx = np.linspace(np.min(coherence_set), np.max(coherence_set),100)
plt.plot(nx, cumulative_gaussian(*res.x,nx),'k') #plot psychometric func

for i,e in zip(coherence_set,ci_right): # plot data and confidence intervals
    plt.plot(i*np.array([1,1]),e,'_-',lw=.5,color='black')
plt.plot(coherence_set,p_right, 'ko', markerfacecolor='lightgray',markersize=6)

plt.vlines(0,0,1,color = 'k',lw = 0.3)
plt.hlines(0.5,np.min(coherence_set),np.max(coherence_set),color = 'k',lw = 0.3)
plt.ylabel('$P_{right}$')
plt.xlabel('Coherence')
separate_axes(plt.gca())
if SAVE_FIGS:
    #plt.savefig(FIGSAVE_PATH / 'psychometric_function.pdf', dpi=500, bbox_inches='tight', format='pdf')
    plt.savefig(FIGSAVE_PATH / 'psychometric_function2.pdf', dpi=500, bbox_inches='tight', format='pdf')

In [ ]:
# Plot a curve for each mouse
nx = np.linspace(np.min(coherence_set), np.max(coherence_set),100)

mice = np.unique(f.session_animals)
for m in mice:
    coherence = [-inpt[:,0] for inpt,animal in zip(f.inpts,f.session_animals) if animal == m]
    coherence = np.hstack(coherence)
    coherence = coherence * 20
    choices = [c.squeeze() for c,animal in zip(f.outputs,f.session_animals) if animal == m]
    choices = np.hstack(choices)

    func = lambda pars: neg_log_likelihood_error(cumulative_gaussian, pars, coherence.astype(float),choices.astype(float))
    # x0 is the initial guess for the fit, it is an important parameter
    x0 = [0.,0.1,p_right[0],1 - p_right[-1]]

    coherence_set = np.unique(coherence)
    p_right, ci_right = compute_p_right(coherence, choices)
    bounds = [(coherence_set[0],coherence_set[-1]),(0.0001,10),(0,0.7),(0,.7)]

    #for i,e in zip(coherence_set,ci_right): # plot data and confidence intervals
    #    plt.plot(i*np.array([1,1]),e,'_-',lw=.5,color='black')
    #plt.plot(coherence_set,p_right, 'ko', markerfacecolor='lightgray',markersize=6)

    res = minimize(func, x0, options = dict(maxiter = 500*len(x0),adaptive=True),
                   bounds = bounds, method='Nelder-Mead') # method = 'L-BFGS-B',
    plt.plot(nx, cumulative_gaussian(*res.x,nx),'gray')

func = lambda pars: neg_log_likelihood_error(cumulative_gaussian, pars, all_coherence.astype(float),all_choices.astype(float))
res = minimize(func, x0, options = dict(maxiter = 500*len(x0),adaptive=True),
               bounds = bounds, method='Nelder-Mead') # method = 'L-BFGS-B',
plt.plot(nx, cumulative_gaussian(*res.x,nx),'k') #plot psychometric func for all mice

plt.vlines(0,0,1,color = 'k',lw = 0.3)
plt.hlines(0.5,np.min(coherence_set),np.max(coherence_set),color = 'k',lw = 0.3)
plt.ylabel('$P_{right}$')
plt.xlabel('Coherence')
separate_axes(plt.gca())
if SAVE_FIGS:
    plt.savefig(FIGSAVE_PATH / 'psychometric_function_individual_mice.pdf', dpi=500, bbox_inches='tight', format='pdf')


In [ ]:
# Plot a curve for each mouse
nx = np.linspace(np.min(coherence_set), np.max(coherence_set),100)
colors = ["#77AC30", "#D95319", "#EDB120", "#7E2F8E"]
colors = ["#D95319", "#EDB120",  "#7E2F8E", "#77AC30",'#FF1493']


mice = np.unique(f.session_animals)
all_pars = []
#mice = ['mSM63']
for mi,m in enumerate(mice):
    coherence = [-inpt[:,0] for inpt,animal in zip(f.inpts,f.session_animals) if animal == m]
    coherence = np.hstack(coherence) 
    coherence = coherence * 20
    choices = [c.squeeze() for c,animal in zip(f.outputs,f.session_animals) if animal == m]
    choices = np.hstack(choices)

    func = lambda pars: neg_log_likelihood_error(cumulative_gaussian, pars, coherence.astype(float),choices.astype(float))
    # x0 is the initial guess for the fit, it is an important parameter

    coherence_set = np.unique(coherence)
    p_right, ci_right = compute_p_right(coherence, choices)

    x0 = [0.,0.1,p_right[0],1 - p_right[-1]]
    bounds = [(-5,5),(0.001,10000),(0,.5),(0,.5)]

    print(bounds)
    print(x0)

    for i,e in zip(coherence_set,ci_right): # plot data and confidence intervals
        plt.plot(i*np.array([1,1]),e,'_-',lw=.5,color='black')
    plt.plot(coherence_set,p_right, 'ko', markerfacecolor=colors[mi],markersize=6)

    res = minimize(func, x0, options = dict(maxiter = 500*len(x0),adaptive=True),
                   bounds = bounds,)# method='L-BFGS-B') # method = 'L-BFGS-B',
    all_pars.append(res.x)
    plt.plot(nx, cumulative_gaussian(*res.x,nx),color=colors[mi])
all_pars = np.array(all_pars)


plt.vlines(0,0,1,color = 'k',lw = 0.3)
plt.hlines(0.5,np.min(coherence_set),np.max(coherence_set),color = 'k',lw = 0.3)
plt.ylabel('$P_{right}$')
plt.xlabel('Coherence')
separate_axes(plt.gca())
if True:
    plt.savefig(FIGSAVE_PATH / 'psychometric_function_individual_mice_2.pdf', dpi=500, bbox_inches='tight', format='pdf')


In [ ]:
high_lapses, low_lapses = [],[]
for p in all_pars:
    func = lambda x: cumulative_gaussian(*p,x)
    low_lapses.append(func(-20))
    high_lapses.append(1 - func(20))

plotting_pars = all_pars
plotting_pars[:,2] = low_lapses
plotting_pars[:,3] = high_lapses

In [ ]:
# iterate over params and plot them
fig, axs = plt.subplots(1,4, figsize=(10,5))
lims = [(-1.2,1.2),(0,.1),(-.01,0.1),(-.01,0.1)]
for i,ax in enumerate(axs):
    ps = plotting_pars[:,i]
    #ps = all_pars[i,:]
    mean_value = np.mean(ps)
    print(ps)
    sem = np.std(ps)/np.sqrt(len(ps))
    ax.bar(0,mean_value, yerr=sem, width=1, color='grey', capsize=3)
    shift = np.random.normal(0,0.0,len(ps))
    ax.scatter(np.zeros(len(ps)) + shift, ps, alpha=1, color=colors)
    ax.set_title(['bias','slope','lapse low','lapse high'][i])
    ax.set_xlim(-1,1)
    #ax.set_ylim(lims[i])
    ax.set_xticks([])
    ax.spines[['top','right','bottom']].set_visible(False)
plt.tight_layout()

if True:
    plt.savefig(FIGSAVE_PATH / 'psychometric_function_individual_mice_params.pdf', dpi=500, bbox_inches='tight', format='pdf')

## Now repeat this but plot over different states

In [ ]:
mouse_trials = []
for mouse,inpt in zip(f.session_animals, f.inpts):
    nt = inpt.shape[0]
    m = [mouse] * nt
    mouse_trials.extend(m)
mouse_trials = np.array(mouse_trials)

In [ ]:
max_state = np.argmax(states, axis=1)

pr, cr = [],[]
models = []

for i in range(states.shape[1]):
    state_choices = all_choices[max_state == i]
    state_coherence = all_coherence[max_state == i]

    #state_choices = all_choices[(max_state == i) & (mouse_trials == 'Fez71')]
    #state_coherence = all_coherence[(max_state == i) & (mouse_trials == 'Fez71')]


    p_right, ci_right = compute_p_right(state_coherence, state_choices)
    print(f'There are {len(state_choices)} trials in state {i}')
    pr.append(p_right)
    cr.append(ci_right)
    
    func = lambda pars: neg_log_likelihood_error(cumulative_gaussian, pars, state_coherence.astype(float),state_choices.astype(float))
    res = minimize(func, x0, options = dict(maxiter = 500*len(x0),adaptive=True),
                   bounds = bounds, method='Nelder-Mead') # method = 'L-BFGS-B',
    print(f'params are {res.x}')
    models.append(res)

#alpha, beta, gamma, lambda


In [ ]:
# plot fractional occupancy
state_inds = np.unique(max_state)
frac_occ = []
for i in state_inds:
    frac_occ.append(np.sum(max_state == i) / len(max_state))
print(frac_occ)
plt.bar(np.arange(len(state_inds)), frac_occ, width=.8)
plt.ylabel('Fractional Occupancy')
separate_axes(plt.gca())
if SAVE_FIGS:
    plt.savefig(FIGSAVE_PATH / 'fractional_occupancy.pdf', dpi=500, bbox_inches='tight', format='pdf')
#plt.savefig(FIGSAVE_PATH / 'fractional_occupancy.pdf', dpi=500, bbox_inches='tight', format='pdf')
plt.show()

# now do it by mice 
#for m in np.unique(mouse_trials):
#    mouse_max_state = max_state[mouse_trials == m]
#    frac_occ = []
#    for i in state_inds:
#        frac_occ.append(np.sum(mouse_max_state == i) / len(mouse_max_state))
#    print(frac_occ)
#    plt.bar(np.arange(len(state_inds)), frac_occ, width=.8)
#    plt.ylabel('Fractional Occupancy')
#    separate_axes(plt.gca())
#    plt.show()



In [ ]:
# ploting
linecols = ['black','red','blue','pink','green']
labs = ['Engaged','Left Bias','Right Bias','Extra1','Extra2']

for i in range(states.shape[1]):
    plt.plot(nx, cumulative_gaussian(*models[i].x,nx),color=linecols[i],label=labs[i]) #plot psychometric func

    for j,e in zip(coherence_set,cr[i]): # plot data and confidence intervals
        plt.plot(j*np.array([1,1]),e,'_-',lw=.5,color='black')
    plt.plot(coherence_set,pr[i], 'ko', markerfacecolor=linecols[i],markersize=6)

plt.vlines(0,0,1,color = 'k',lw = 0.3)
plt.legend()
plt.hlines(0.5,np.min(coherence_set),np.max(coherence_set),color = 'k',lw = 0.3)
plt.ylabel('$P_{right}$')
plt.xlabel('Coherence')
separate_axes(plt.gca())
if SAVE_FIGS:
    #plt.savefig(FIGSAVE_PATH / 'psychometric_function_over_states.pdf', dpi=500, bbox_inches='tight', format='pdf')
    plt.savefig(FIGSAVE_PATH / 'psychometric_function_over_states2.pdf', dpi=500, bbox_inches='tight', format='pdf')

### Plot weights and example state occupancies

In [ ]:
weights = f.return_ordered_weights(modelindex=ind)
#weights = mdl.observations.params
weights = np.squeeze(weights)
for i in range(weights.shape[0]):
    plt.plot(weights[i,:], '-o',color=linecols[i])

plt.xticks(range(len(f.input_terms_list)))
plt.gca().set_xticklabels(f.input_terms_list)
#plt.xticks(range(2))
#plt.gca().set_xticklabels(['Coherence','Bias'])
#plt.xticks(range(4), rotation=-45)
#plt.gca().set_xticklabels(['Coherence','Bias','Previous choice', 'Win-stay lose-switch'])
plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.legend(['Engaged State','Left Bias State','Right Bias State'], loc='upper right')
#plt.title(f'Weights for merged sessions: {n_states} states')
plt.ylabel('GLM Weight')
separate_axes(plt.gca())
if SAVE_FIGS:
    plt.savefig(FIGSAVE_PATH / 'weights.pdf', dpi=500, bbox_inches='tight', format='pdf')

In [ ]:
SESSION_IND = 47

states = f.return_ordered_states(modelindex=ind)
sess = states[SESSION_IND]
plt.figure(dpi=500)
for i in range(weights.shape[0]):
    plt.plot(sess[:,i],color=linecols[i])
separate_axes(plt.gca())
plt.xlabel('Trial')
plt.ylabel('P(State)')
if SAVE_FIGS:
    plt.savefig(FIGSAVE_PATH / 'recovered_states.pdf', dpi=500, bbox_inches='tight', format='pdf')


In [ ]:
plt.figure(figsize=(10,3))
x = np.arange(2000)
locs = np.array([320, 500, 1100, 1200, 1500])
y = np.where(np.isin(x,locs), .2, 0)
plt.plot(x,y, color='black')
#plt.savefig(FIGSAVE_PATH / 'line.pdf', dpi=500, bbox_inches='tight', format='pdf')

plt.figure(figsize=(10,3))
x = np.arange(2000)
locs = np.array([120, 300, 560,1100, 900, 1500,1540])
y = np.where(np.isin(x,locs), .2, 0)
plt.plot(x,y, color='black')
#plt.savefig(FIGSAVE_PATH / 'line2.pdf', dpi=500, bbox_inches='tight', format='pdf')

plt.figure(figsize=(10,3))
locs = np.array([500, 750])
y = np.where(np.isin(x,locs), .2, 0)
plt.plot(x,y, color='black')
#plt.savefig(FIGSAVE_PATH / 'line3.pdf', dpi=500, bbox_inches='tight', format='pdf')